# Exploratory data analysis: recommendation engine data

This notebook explores the synthetic users, items, and ratings datasets to understand rating patterns, user behavior, and item characteristics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

users = pd.read_csv("../data/users.csv")
items = pd.read_csv("../data/items.csv")
ratings = pd.read_csv("../data/ratings.csv")

print(f"Users: {users.shape}")
print(f"Items: {items.shape}")
print(f"Ratings: {ratings.shape}")

## Data overview

In [ ]:
print("Users columns:", list(users.columns))
print("Items columns:", list(items.columns))
print("Ratings columns:", list(ratings.columns))
print(f"\nMissing values:")
print(f"  Users: {users.isnull().sum().sum()}")
print(f"  Items: {items.isnull().sum().sum()}")
print(f"  Ratings: {ratings.isnull().sum().sum()}")
print(f"\nRating range: {ratings['rating'].min()} - {ratings['rating'].max()}")
print(f"Unique users: {ratings['user_id'].nunique()}, Unique items: {ratings['item_id'].nunique()}")

total_possible = users.shape[0] * items.shape[0]
sparsity = 1 - len(ratings) / total_possible
print(f"Sparsity: {sparsity:.2%}")

## Rating distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rating_counts = ratings["rating"].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, color="#3B6FD4", edgecolor="black")
axes[0].set_title("Rating distribution")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")
axes[0].set_xticks([1, 2, 3, 4, 5])
for i, v in zip(rating_counts.index, rating_counts.values):
    axes[0].text(i, v + 100, f"{v:,}", ha="center", fontweight="bold")

axes[1].pie(rating_counts.values, labels=[f"{r} star" for r in rating_counts.index],
            colors=["#ef4444", "#f59e0b", "#a3a3a3", "#3B6FD4", "#22c55e"],
            autopct="%1.1f%%", startangle=90)
axes[1].set_title("Rating proportions")

plt.tight_layout()
plt.show()

## Ratings per user

In [ ]:
ratings_per_user = ratings.groupby("user_id").size()
ratings_per_item = ratings.groupby("item_id").size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ratings_per_user, bins=40, color="#3B6FD4", edgecolor="black", alpha=0.8)
axes[0].set_title("Ratings per user")
axes[0].set_xlabel("Number of ratings")
axes[0].set_ylabel("Number of users")
axes[0].axvline(ratings_per_user.mean(), color="#E8C230", linestyle="--", label=f"Mean: {ratings_per_user.mean():.1f}")
axes[0].legend()

axes[1].hist(ratings_per_item, bins=40, color="#E8C230", edgecolor="black", alpha=0.8)
axes[1].set_title("Ratings per item")
axes[1].set_xlabel("Number of ratings")
axes[1].set_ylabel("Number of items")
axes[1].axvline(ratings_per_item.mean(), color="#3B6FD4", linestyle="--", label=f"Mean: {ratings_per_item.mean():.1f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Ratings per user -- min: {ratings_per_user.min()}, max: {ratings_per_user.max()}, median: {ratings_per_user.median():.0f}")
print(f"Ratings per item -- min: {ratings_per_item.min()}, max: {ratings_per_item.max()}, median: {ratings_per_item.median():.0f}")

## User demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

users["age_group"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#3B6FD4", edgecolor="black")
axes[0].set_title("Users by age group")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

users["region"].value_counts().plot(kind="bar", ax=axes[1], color="#E8C230", edgecolor="black")
axes[1].set_title("Users by region")
axes[1].set_ylabel("Count")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha="right")

axes[2].hist(users["signup_months_ago"], bins=30, color="#3B6FD4", edgecolor="black", alpha=0.8)
axes[2].set_title("Signup recency")
axes[2].set_xlabel("Months since signup")
axes[2].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Item characteristics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

items["category"].value_counts().plot(kind="barh", ax=axes[0], color="#3B6FD4", edgecolor="black")
axes[0].set_title("Items per category")
axes[0].set_xlabel("Count")

items["price_tier"].value_counts().plot(kind="bar", ax=axes[1], color="#E8C230", edgecolor="black")
axes[1].set_title("Items by price tier")
axes[1].set_ylabel("Count")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

axes[2].hist(items["avg_rating"], bins=25, color="#3B6FD4", edgecolor="black", alpha=0.8)
axes[2].set_title("Item average rating distribution")
axes[2].set_xlabel("Average rating")
axes[2].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Average rating by category

In [ ]:
merged = ratings.merge(items[["item_id", "category", "price_tier"]], on="item_id")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

avg_by_cat = merged.groupby("category")["rating"].mean().sort_values(ascending=True)
avg_by_cat.plot(kind="barh", ax=axes[0], color="#E8C230", edgecolor="black")
axes[0].set_title("Average rating by category")
axes[0].set_xlabel("Average rating")
axes[0].set_xlim(0, 5)

avg_by_price = merged.groupby("price_tier")["rating"].mean().sort_values(ascending=True)
avg_by_price.plot(kind="barh", ax=axes[1], color="#3B6FD4", edgecolor="black")
axes[1].set_title("Average rating by price tier")
axes[1].set_xlabel("Average rating")
axes[1].set_xlim(0, 5)

plt.tight_layout()
plt.show()

## Rating patterns by user demographics

In [ ]:
user_ratings = ratings.merge(users, on="user_id")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

user_ratings.boxplot(column="rating", by="age_group", ax=axes[0])
axes[0].set_title("Rating by age group")
axes[0].set_xlabel("Age group")
axes[0].set_ylabel("Rating")
plt.sca(axes[0])
plt.title("Rating by age group")

user_ratings.boxplot(column="rating", by="region", ax=axes[1])
axes[1].set_title("Rating by region")
axes[1].set_xlabel("Region")
axes[1].set_ylabel("Rating")
plt.sca(axes[1])
plt.title("Rating by region")

plt.suptitle("")
plt.tight_layout()
plt.show()

## Sparsity visualization

In [ ]:
# Visualize a small sample of the user-item matrix
from scipy.sparse import csr_matrix

sample_users = sorted(ratings["user_id"].unique())[:50]
sample_items = sorted(ratings["item_id"].unique())[:100]
sample_ratings = ratings[
    (ratings["user_id"].isin(sample_users)) & (ratings["item_id"].isin(sample_items))
]

pivot = sample_ratings.pivot_table(index="user_id", columns="item_id", values="rating")

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot.fillna(0), cmap="YlOrRd", ax=ax, cbar_kws={"label": "Rating"})
ax.set_title("User-item rating matrix (50 users x 100 items sample)")
ax.set_xlabel("Item ID")
ax.set_ylabel("User ID")
plt.tight_layout()
plt.show()

filled = pivot.notna().sum().sum()
total = pivot.shape[0] * pivot.shape[1]
print(f"Sample sparsity: {1 - filled/total:.2%}")

## Correlation analysis

In [ ]:
item_stats = ratings.groupby("item_id").agg(
    user_rating_count=("rating", "count"),
    user_avg_rating=("rating", "mean"),
    user_std_rating=("rating", "std"),
).reset_index()
item_stats = item_stats.merge(items, on="item_id")

numeric_cols = ["user_rating_count", "user_avg_rating", "avg_rating", "num_ratings"]
corr = item_stats[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation heatmap (item-level features)")
plt.tight_layout()
plt.show()

## Long tail distribution

In [ ]:
# Item popularity long tail
item_pop = ratings.groupby("item_id").size().sort_values(ascending=False).reset_index()
item_pop.columns = ["item_id", "count"]
item_pop["rank"] = range(1, len(item_pop) + 1)
item_pop["cumulative_pct"] = item_pop["count"].cumsum() / item_pop["count"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(item_pop["rank"], item_pop["count"], color="#3B6FD4", alpha=0.7, width=1.0)
axes[0].set_title("Item popularity (long tail)")
axes[0].set_xlabel("Item rank")
axes[0].set_ylabel("Number of ratings")

axes[1].plot(item_pop["rank"], item_pop["cumulative_pct"], color="#E8C230", linewidth=2)
axes[1].axhline(y=0.8, color="#ef4444", linestyle="--", label="80% of ratings")
top_20_items = int(len(item_pop) * 0.2)
axes[1].axvline(x=top_20_items, color="#3B6FD4", linestyle="--", label=f"Top 20% items ({top_20_items})")
axes[1].set_title("Cumulative rating distribution")
axes[1].set_xlabel("Item rank")
axes[1].set_ylabel("Cumulative fraction of ratings")
axes[1].legend()

plt.tight_layout()
plt.show()

pct_80 = item_pop[item_pop["cumulative_pct"] <= 0.8].shape[0]
print(f"{pct_80} items ({pct_80/len(item_pop):.1%}) account for 80% of all ratings")

## Summary

Key takeaways from the exploratory analysis:

1. **Dataset contains 2,000 users, 500 items, and ~50K ratings** with ~95% sparsity, typical for recommendation systems
2. **Rating distribution is centered around 3-4 stars** with a slight skew toward higher ratings
3. **User activity follows a power-law pattern** -- a minority of users contribute a large fraction of ratings
4. **Item popularity shows a long tail** -- a small number of items receive most ratings while many items have few
5. **10 item categories** are represented with Electronics, Books, and Clothing being the most common
6. **Rating patterns are consistent across user demographics** -- no strong bias by age group or region